# VarChat — Ölçüm defteri (Google Colab)

İki modeli **aynı ortamda, aynı donmuş test setiyle** ölçer:

1. **Ham `qwen2.5:7b`**: taban model
2. **`varchat`**: `qwen2.5:7b` + eğitilmiş LoRA adaptörü (Drive'da `varchat-lora/varchat-lora.gguf` varsa)

Her model için önce 56 girdinin özeti üretilir (`b06`), sonra her cümle kaynağına karşı yargıç modelle denetlenir (`b07`).
Yargıç iki modelde de ham `qwen2.5:7b`'dir; böylece ikisi aynı ölçüyle puanlanır.

**Kullanım:** *Runtime → Change runtime type → **GPU** (T4 yeter, L4 daha hızlı)* → *Runtime → **Run all***.
2. hücrede Drive izin penceresi açılınca hesabını seçip onayla.

- Sonuçlar Drive'da **`varchat-olcum/sonuclar`** klasörüne yazılır. Oturum koparsa defteri yeniden *Run all* yap: kaldığı yerden devam eder.
- Tarayıcı sekmesini açık tut; Colab Pro arka planda çalışmayı garanti etmez.
- Eğitim henüz yapılmadıysa 6. bölüm kendiliğinden atlanır. Eğitimden sonra defteri yeniden *Run all* yapman yeterli: ham model tekrar ölçülmez.

## 0) GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1) Projeyi getir + paketler

In [ ]:
import os
if not os.path.exists('/content/varchat'):
    !git clone -q https://github.com/atu-ce/varchat.git /content/varchat
else:
    !git -C /content/varchat pull -q || echo "pull atlandı (yerel değişiklik var); taze kod için Runtime → Disconnect and delete runtime"
%cd /content/varchat
!git log --oneline -1
# Yereldeki sürümlerle aynı istemci; requests Colab'ın kendi sürümü kalır (Colab paketleri ona bağlı)
!pip install -q "ollama==0.6.2" "rapidfuzz==3.14.5"

## 2) Drive'a bağlan: sonuçlar kalıcı olsun

Projedeki `degerlendirme/sonuclar` klasörü Drive'daki `varchat-olcum/sonuclar` klasörüne bağlanır.
Doğru cevap listesi ve donmuş arama kaydı ilk seferde Drive'a kopyalanır; üretim ve sadakat dosyaları doğrudan Drive'a yazılır.

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

SONUC = '/content/drive/MyDrive/varchat-olcum/sonuclar'
LORA  = '/content/drive/MyDrive/varchat-lora/varchat-lora.gguf'     # eğitim defterinin yazdığı adaptör
os.makedirs(SONUC, exist_ok=True)

yerel = '/content/varchat/degerlendirme/sonuclar'
if not os.path.islink(yerel):
    for ad in os.listdir(yerel):
        kaynak = os.path.join(yerel, ad)
        if os.path.isfile(kaynak) and not os.path.exists(os.path.join(SONUC, ad)):
            shutil.copy2(kaynak, SONUC)          # Drive'da olanın üstüne yazılmaz
    shutil.rmtree(yerel)
    os.symlink(SONUC, yerel)
print("sonuçlar ->", SONUC)
print(sorted(os.listdir(yerel)))

## 3) Ollama'yı kur, başlat, taban modeli indir

Yereldeki uygulamayla aynı model: `qwen2.5:7b` (4-bit, Q4_K_M). Ortam bilgisi (GPU, Ollama sürümü, model kimliği) tez için Drive'a yazılır.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests, json, datetime

def ollama_hazir():
    try:
        return requests.get('http://127.0.0.1:11434/api/version', timeout=2).json()
    except Exception:
        return None

if not ollama_hazir():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_hazir():
            break
print("Ollama:", ollama_hazir())
assert ollama_hazir(), "Ollama başlamadı; /content/ollama.log dosyasına bak"

In [ ]:
!ollama pull qwen2.5:7b
!ollama run qwen2.5:7b "Merhaba. Tek kelimeyle cevap ver."
ps_cikti = !ollama ps
print("\n".join(ps_cikti))
if not any("GPU" in s for s in ps_cikti[1:]):
    print("\nUYARI: model GPU'da değil. Runtime type GPU mu? Değilse değiştirip baştan Run all yap.")

gpu = !nvidia-smi --query-gpu=name --format=csv,noheader
liste = !ollama list
kod_surumu = !git -C /content/varchat log --oneline -1
ortam = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "gpu": list(gpu), "ollama": ollama_hazir(),
         "modeller": list(liste), "kod": list(kod_surumu)}
json.dump(ortam, open(f"{SONUC}/ortam_{datetime.date.today().isoformat()}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps(ortam, ensure_ascii=False, indent=1))

## 4) Hızlı kontrol: denetim testleri (model çağırmaz, birkaç saniye)

In [ ]:
!python degerlendirme/denetim_testi.py | tail -1

## 5) Taban model: ham `qwen2.5:7b`

Önce özetler üretilir, sonra her atıflı cümle kaynağına karşı yargılanır. Kaldığı yerden devam eder.

In [ ]:
!python -u degerlendirme/b06_toplu_uret.py qwen2.5:7b

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen2.5:7b

## 6) Eğitilmiş model: `varchat` (7B + LoRA)

Önce `fine_tune/colab_egitim.ipynb` çalışmış ve Drive'a `varchat-lora/varchat-lora.gguf` yazmış olmalı; yoksa bu bölüm atlanır.
Adaptör, eğitimdeki taban modelin Ollama sürümü olan `qwen2.5:7b`'nin üstüne bindirilir.

In [ ]:
LORA_VAR = os.path.exists(LORA)
if LORA_VAR:
    shutil.copy(LORA, '/content/varchat-lora.gguf')
    with open('/content/Modelfile', 'w') as f:
        f.write("FROM qwen2.5:7b\nADAPTER /content/varchat-lora.gguf\nPARAMETER num_ctx 8192\nPARAMETER temperature 0\n")
    !ollama create varchat -f /content/Modelfile
    !ollama run varchat "Merhaba. Tek kelimeyle cevap ver."
else:
    print("Drive'da", LORA, "yok: eğitim defteri henüz çalışmamış. 6. bölüm atlandı.")

In [ ]:
if LORA_VAR:
    !python -u degerlendirme/b06_toplu_uret.py varchat

In [ ]:
if LORA_VAR:
    !python -u degerlendirme/b07_sadakat.py --model varchat     # yargıç yine ham qwen2.5:7b

## 7) Karşılaştırma tablosu

Katman başına: sadakat (kaynağın desteklediği iddia oranı, yüksek iyi), çelişki ve atıfsız cümle oranı (düşük iyi),
geçersiz atıf, dil çöküşü, kesilen cevap, süre. Tablo Drive'a `karsilastirma_<tarih>.csv` olarak da yazılır.

In [ ]:
import glob, pandas as pd

def son_ozet(model_ad):
    d = glob.glob(f"{SONUC}/sadakat_ozet_{model_ad}_*.json")
    return json.load(open(max(d, key=os.path.getmtime), encoding="utf-8")) if d else None

modeller = {"ham qwen2.5:7b": son_ozet("qwen2.5-7b"), "varchat (7B+LoRA)": son_ozet("varchat")}
OLCULER = [("sadakat", "Sadakat ↑"), ("celiski_orani", "Çelişki oranı ↓"), ("atifsiz_orani", "Atıfsız cümle oranı ↓"),
           ("gecersiz_atif", "Geçersiz atıf ↓"), ("cjk_bozuk", "Dil çöküşü ↓"), ("kesilen", "Kesilen cevap ↓"),
           ("iddia", "Yargılanan iddia"), ("ort_sure_s", "Ortalama süre (sn)")]
satirlar = []
katmanlar = sorted({k for o in modeller.values() if o for k in o["tablo"]}, key=lambda k: (k == "TÜMÜ", k))
for k in katmanlar:
    for anahtar, ad in OLCULER:
        satirlar.append({"katman": k, "ölçü": ad,
                         **{m: (o["tablo"].get(k, {}).get(anahtar) if o else None) for m, o in modeller.items()}})
tablo = pd.DataFrame(satirlar)
tablo.to_csv(f"{SONUC}/karsilastirma_{datetime.date.today().isoformat()}.csv", index=False, encoding="utf-8-sig")
for m, o in modeller.items():
    print(f"{m}: {o['uretim'] if o else 'henüz ölçülmedi'}")
pd.set_option("display.max_rows", 200)
display(tablo)

## 8) Sonuçları bilgisayarına al

Drive'daki **`varchat-olcum/sonuclar`** klasöründen şu dosyaları indirip projede `degerlendirme/sonuclar/` içine koy:
`uretim_*.jsonl`, `sadakat_*.jsonl`, `sadakat_ozet_*.json`, `insan_kontrol_*.csv`, `karsilastirma_*.csv`, `ortam_*.json`.

`insan_kontrol_*.csv`: yargıcın kararlarından rastgele 30 iddia. Son sütunu sen elle doldurursun; yargıcın ne kadar güvenilir
olduğunu (insan-yargıç uyumu) bununla ölçeriz.